In [ ]:
# ============================================================================
# CAD U-Net 3D for Top Brain CT Segmentation - Complete Colab Pipeline
# ============================================================================

#@title Setup & Imports
# If using Colab, uncomment the lines below:
!pip install -q torchio nibabel tqdm matplotlib
# from google.colab import drive
# drive.mount('/content/drive')

import os, glob, random, math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import nibabel as nib
from tqdm import tqdm

try:
    import torchio as tio
except Exception:
    print("TorchIO not available. Install with `pip install torchio` for 3D augmentations.")
    tio = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.5/199.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 23.1 MB/s eta 0:00:00
Device: cuda


In [ ]:
#@title Configuration
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation"
IMG_GLOB = "imagesTr_topbrain_ct/*.nii.gz"
MSK_GLOB = "labelsTr_topbrain_ct/*.nii.gz"
LABELMAP_TXT = "/content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation/labelmap_topbrain_ct.txt"

# Classes
NUM_CLASSES = 41   # 0..40 (0 = background)
BG_INDEX = 0

# Model & training
PATCH_SIZE = (64, 128, 128)  # (D, H, W) for 3D patches
BASE_CH = 32                  # Base channels for CAD U-Net 3D
BATCH_SIZE = 2                # Small batch size for 3D (memory intensive)
EPOCHS = 80
LR_MAX = 1e-3                 # OneCycle peak LR
WEIGHT_DECAY = 1e-5
ACCUM_STEPS = 2               # Gradient accumulation steps

# Loss weights
ALPHA_CE = 1.0                # CrossEntropy weight
BETA_DICE = 1.0               # Dice weight

# Sampling
FOREGROUND_SAMPLING_PROB = 0.5
TRAIN_VAL_SPLIT = 0.8         # 80% train, 20% validation

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# Results
RESULTS_DIR = "/content/drive/MyDrive/3D_CAD_Unet/Results"
Checkpoint = "/content/drive/MyDrive/3D_CAD_Unet/Model"
CHECKPOINT_PATH = os.path.join(Checkpoint, "best_CADUNet3D.pt")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Saving results to:", RESULTS_DIR)


Saving results to: /content/drive/MyDrive/3D_CAD_Unet/Results


In [ ]:
#@title Labelmap Reader + Label Decoding

def read_itksnap_labelmap_txt(path):
    """Read ITK-SNAP labelmap text file."""
    labels = {}
    if not os.path.exists(path):
        print("Labelmap not found:", path, "— proceeding without RGB mapping.")
        return labels
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            try:
                idx = int(parts[0])
            except Exception:
                continue
            entry = {"idx": idx, "name": None, "rgb": None}
            if len(parts) >= 7:
                try:
                    r, g, b = int(parts[3]), int(parts[4]), int(parts[5])
                    entry["rgb"] = (r, g, b)
                except Exception:
                    pass
            labels[idx] = entry
    print(f"Loaded {len(labels)} classes from labelmap file.")
    return labels

LABEL_INFO = read_itksnap_labelmap_txt(LABELMAP_TXT)

def palette_png_to_indexed(mask_img, palette_rgb_to_idx):
    """Convert RGB palette mask to indexed mask."""
    m = np.array(mask_img)
    if m.ndim == 2:
        return m.astype(np.int64)
    if m.ndim == 3 and m.shape[2] == 4:
        m = m[..., :3]
    h, w, _ = m.shape
    out = np.zeros((h * w,), dtype=np.int64)
    flat = m.reshape(-1, 3)
    for i in range(flat.shape[0]):
        col = tuple(int(v) for v in flat[i])
        out[i] = palette_rgb_to_idx.get(col, BG_INDEX)
    return out.reshape(h, w)

def ensure_indexed_mask(mask_array, num_classes=NUM_CLASSES):
    """Ensure mask is indexed format (not RGB)."""
    m = np.array(mask_array)
    if m.ndim == 3 and m.shape[-1] in (3, 4):
        rgb2idx = {}
        for entry in LABEL_INFO.values():
            if entry.get("rgb") is not None:
                rgb2idx[entry["rgb"]] = entry["idx"]
        if not rgb2idx:
            raise ValueError("RGB mask provided but labelmap has no RGB entries.")
        m = palette_png_to_indexed(m, rgb2idx)
    m = m.astype(np.int64)
    m[(m < 0) | (m >= num_classes)] = BG_INDEX
    return m


Loaded 41 classes from labelmap file.


In [ ]:
#@title CAD U-Net 3D Architecture

class SEBlock3D(nn.Module):
    """Squeeze-and-Excitation block for 3D."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Conv3d(ch, ch // r, 1),
            nn.ReLU(inplace=True),
            nn.Conv3d(ch // r, ch, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(self.pool(x))
        return x * w

class SpatialAttn3D(nn.Module):
    """Spatial attention for 3D."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv3d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

def Norm3d(C):
    """Instance normalization for 3D (works better with small batches)."""
    return nn.InstanceNorm3d(C, affine=True, eps=1e-5)

class CADBlock3D(nn.Module):
    """Context-Aware Dilated block with parallel dilated convolutions."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 2

        self.d1 = nn.Sequential(
            nn.Conv3d(in_ch, mid, 3, padding=1, dilation=1),
            Norm3d(mid), nn.ReLU(inplace=True)
        )
        self.d2 = nn.Sequential(
            nn.Conv3d(in_ch, mid // 2, 3, padding=2, dilation=2),
            Norm3d(mid // 2), nn.ReLU(inplace=True)
        )
        self.d3 = nn.Sequential(
            nn.Conv3d(in_ch, mid // 2, 3, padding=4, dilation=4),
            Norm3d(mid // 2), nn.ReLU(inplace=True)
        )
        self.mix = nn.Sequential(
            nn.Conv3d(mid + mid // 2 + mid // 2, out_ch, 1),
            Norm3d(out_ch), nn.ReLU(inplace=True)
        )
        self.se = SEBlock3D(out_ch)
        self.sa = SpatialAttn3D()

    def forward(self, x):
        x1 = self.d1(x)
        x2 = self.d2(x)
        x3 = self.d3(x)
        z = self.mix(torch.cat([x1, x2, x3], dim=1))
        z = self.se(z)
        z = self.sa(z)
        return z

class AttnGate3D(nn.Module):
    """Attention gate for skip connections."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv3d(F_g, F_int, 1), Norm3d(F_int))
        self.W_x = nn.Sequential(nn.Conv3d(F_l, F_int, 1), Norm3d(F_int))
        self.psi = nn.Sequential(nn.Conv3d(F_int, 1, 1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class Up3D(nn.Module):
    """Upsampling block with attention gate."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.attn = AttnGate3D(F_g=out_ch, F_l=skip_ch, F_int=out_ch // 2)
        self.block = nn.Sequential(
            CADBlock3D(out_ch + skip_ch, out_ch),
            CADBlock3D(out_ch, out_ch)
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatch
        dz = skip.size(2) - x.size(2)
        dy = skip.size(3) - x.size(3)
        dx = skip.size(4) - x.size(4)
        if dz or dy or dx:
            x = F.pad(x, (0, max(0, dx), 0, max(0, dy), 0, max(0, dz)))
        skip = self.attn(x, skip)
        x = torch.cat([x, skip], dim=1)
        x = self.block(x)
        return x

class CADUNet3D(nn.Module):
    """CAD U-Net 3D with attention gates and context-aware blocks."""
    def __init__(self, in_ch=1, base=32, num_classes=41):
        super().__init__()
        # Encoder
        self.e1 = nn.Sequential(CADBlock3D(in_ch, base), CADBlock3D(base, base))
        self.p1 = nn.MaxPool3d(2)
        self.e2 = nn.Sequential(CADBlock3D(base, base*2), CADBlock3D(base*2, base*2))
        self.p2 = nn.MaxPool3d(2)
        self.e3 = nn.Sequential(CADBlock3D(base*2, base*4), CADBlock3D(base*4, base*4))
        self.p3 = nn.MaxPool3d(2)
        self.e4 = nn.Sequential(CADBlock3D(base*4, base*8), CADBlock3D(base*8, base*8))
        self.p4 = nn.MaxPool3d(2)

        # Bottleneck
        self.bott = nn.Sequential(CADBlock3D(base*8, base*16), CADBlock3D(base*16, base*16))

        # Decoder
        self.u4 = Up3D(base*16, base*8, base*8)
        self.u3 = Up3D(base*8, base*4, base*4)
        self.u2 = Up3D(base*4, base*2, base*2)
        self.u1 = Up3D(base*2, base, base)

        # Output head
        self.head_mc = nn.Conv3d(base, num_classes, 1)

    def forward(self, x):
        c1 = self.e1(x)
        p1 = self.p1(c1)
        c2 = self.e2(p1)
        p2 = self.p2(c2)
        c3 = self.e3(p2)
        p3 = self.p3(c3)
        c4 = self.e4(p3)
        p4 = self.p4(c4)

        b = self.bott(p4)

        x = self.u4(b, c4)
        x = self.u3(x, c3)
        x = self.u2(x, c2)
        x = self.u1(x, c1)

        return self.head_mc(x)

# Test model
print("\n" + "="*60)
print("Testing CAD U-Net 3D architecture...")
print("="*60)
with torch.no_grad():
    test_model = CADUNet3D(in_ch=1, base=BASE_CH, num_classes=NUM_CLASSES).to(device)
    test_input = torch.randn(1, 1, 32, 64, 64, device=device)
    test_output = test_model(test_input)
    print(f"✓ Model initialized successfully")
    print(f"  Input shape:  {tuple(test_input.shape)}")
    print(f"  Output shape: {tuple(test_output.shape)}")
    print(f"  Parameters: {sum(p.numel() for p in test_model.parameters()):,}")
    del test_model, test_input, test_output
    torch.cuda.empty_cache()


Testing CAD U-Net 3D architecture...
✓ Model initialized successfully
  Input shape:  (1, 1, 32, 64, 64)
  Output shape: (1, 41, 32, 64, 64)
  Parameters: 23,677,091


In [ ]:
#@title Loss Functions

class DiceLossMultiClass(nn.Module):
    """Multi-class Dice loss."""
    def __init__(self, smooth=1.0, eps=1e-7, weight=None, ignore_index=None, reduction="mean"):
        super().__init__()
        self.smooth = smooth
        self.eps = eps
        self.register_buffer("weight", weight if weight is not None else None)
        self.ignore_index = ignore_index
        assert reduction in ("mean", "sum", "none")
        self.reduction = reduction

    def forward(self, logits, target):
        B, C = logits.shape[:2]
        probs = F.softmax(logits, dim=1)

        with torch.no_grad():
            target_oh = F.one_hot(target.long(), num_classes=C)
            dims = list(range(1, target_oh.dim()-1))
            target_oh = target_oh.permute(0, -1, *dims).float()

            if self.ignore_index is not None:
                valid = (target != self.ignore_index).unsqueeze(1).float()
                target_oh = target_oh * valid
                probs = probs * valid

        probs_f = probs.reshape(B, C, -1)
        tgt_f = target_oh.reshape(B, C, -1)

        inter = (probs_f * tgt_f).sum(-1).sum(0)
        card = (probs_f + tgt_f).sum(-1).sum(0)

        dice = (2 * inter + self.smooth) / (card + self.smooth + self.eps)
        loss_c = 1.0 - dice

        if self.weight is not None:
            if self.weight.numel() != logits.size(1):
                raise ValueError("Dice weight length must equal num classes")
            loss_c = loss_c * self.weight

        if self.reduction == "mean":
            return loss_c.mean()
        if self.reduction == "sum":
            return loss_c.sum()
        return loss_c

class CombinedLossMultiClass(nn.Module):
    """Combined Cross-Entropy + Dice loss."""
    def __init__(self, alpha=1.0, beta=1.0, dice_kwargs=None, ce_weight=None, ignore_index=None):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

        # CrossEntropyLoss requires ignore_index to be int, not None
        if ignore_index is None:
            self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        else:
            self.ce = nn.CrossEntropyLoss(weight=ce_weight, ignore_index=ignore_index)

        dice_kwargs = dice_kwargs or {}
        if 'ignore_index' not in dice_kwargs:
            dice_kwargs['ignore_index'] = ignore_index
        self.dice = DiceLossMultiClass(**dice_kwargs)

    def forward(self, logits, target):
        return self.alpha * self.ce(logits, target.long()) + self.beta * self.dice(logits, target)

print("✓ Loss functions defined")

✓ Loss functions defined


In [ ]:
#@title Dataset Class

class TopBrainCT3D(Dataset):
    """Dataset for 3D Top Brain CT segmentation with patch sampling."""
    def __init__(self, img_files, msk_files,
                 patch_size=(64, 128, 128),
                 augment=True,
                 foreground_sampling=True,
                 num_classes=41,
                 cache_volumes=True,
                 patches_per_volume=10):
        assert len(img_files) == len(msk_files), "Images vs masks mismatch"
        self.img_files = img_files
        self.msk_files = msk_files
        self.patch_size = patch_size
        self.augment = augment and (tio is not None)
        self.fg_sampling = foreground_sampling
        self.num_classes = num_classes
        self.cache_volumes = cache_volumes
        self.patches_per_volume = patches_per_volume
        self._cache = {}

        if self.augment:
            self.transforms = tio.Compose([
                tio.RandomFlip(axes=('LR',), flip_probability=0.5),
                tio.RandomAffine(scales=(0.9, 1.1), degrees=5, isotropic=True, p=0.5),
                tio.RandomElasticDeformation(num_control_points=7, max_displacement=5, p=0.3),
                tio.RandomGamma(log_gamma=(-0.3, 0.3), p=0.3),
            ])
        else:
            self.transforms = None

    def _load_volume(self, idx):
        """Load and cache volume."""
        if self.cache_volumes and idx in self._cache:
            return self._cache[idx]

        vol = nib.load(self.img_files[idx]).get_fdata().astype(np.float32)
        msk = nib.load(self.msk_files[idx]).get_fdata().astype(np.int16)

        # Normalize volume
        vol = (vol - vol.mean()) / (vol.std() + 1e-6)

        # Ensure mask is indexed
        msk = ensure_indexed_mask(msk, self.num_classes)

        if self.cache_volumes:
            self._cache[idx] = (vol, msk)
        return vol, msk

    def _sample_patch(self, vol, msk):
        """Sample a patch from volume, preferring foreground if enabled."""
        D, H, W = vol.shape
        pd, ph, pw = self.patch_size

        attempts = 5 if self.fg_sampling else 1
        for _ in range(attempts):
            dz = random.randint(0, max(0, D - pd))
            dy = random.randint(0, max(0, H - ph))
            dx = random.randint(0, max(0, W - pw))

            sub = msk[dz:dz+pd, dy:dy+ph, dx:dx+pw]
            if not self.fg_sampling or np.any(sub > 0):
                break

        return vol[dz:dz+pd, dy:dy+ph, dx:dx+pw], sub

    def __len__(self):
        return len(self.img_files) * self.patches_per_volume

    def __getitem__(self, idx):
        vidx = idx % len(self.img_files)
        vol, msk = self._load_volume(vidx)
        v, m = self._sample_patch(vol, msk)

        # Apply augmentations if enabled
        if self.transforms is not None:
            # FIXED: TorchIO expects 4D tensors (C, D, H, W), not 5D
            # Convert numpy arrays to 4D tensors by adding channel dimension
            v_tensor = torch.from_numpy(v).float().unsqueeze(0)  # (1, D, H, W) 4D
            m_tensor = torch.from_numpy(m).long().unsqueeze(0)    # (1, D, H, W) 4D

            # Create TorchIO subject with 4D tensors
            subj = tio.Subject(
                image=tio.ScalarImage(tensor=v_tensor),  # 4D input
                mask=tio.LabelMap(tensor=m_tensor)        # 4D input
            )

            # Apply transforms
            out = self.transforms(subj)

            # Extract back to numpy, remove channel dim
            v = out.image.data[0].numpy()  # (D, H, W) 3D
            m = out.mask.data[0].numpy()   # (D, H, W) 3D

        # Return final tensors
        # Model expects (B, C, D, H, W) 5D - DataLoader adds batch dimension
        img = torch.from_numpy(v).float().unsqueeze(0)  # (1, D, H, W) 4D
        mask = torch.from_numpy(m).long()                # (D, H, W) 3D
        return img, mask

print("✓ Dataset class defined")

✓ Dataset class defined


In [ ]:
#@title Training & Validation Functions

def train_one_epoch(model, loader, optimizer, criterion, device, accum_steps=1):
    """Train for one epoch with gradient accumulation."""
    model.train()
    scaler = GradScaler()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc="Training", leave=False)
    for batch_idx, (imgs, masks) in enumerate(pbar):
        imgs, masks = imgs.to(device), masks.to(device)

        with autocast():
            logits = model(imgs)
            loss = criterion(logits, masks) / accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accum_steps
        pbar.set_postfix({'loss': f'{loss.item() * accum_steps:.4f}'})

    return total_loss / max(1, len(loader))

@torch.no_grad()
def validate(model, loader, device, num_classes=41):
    """Validate and compute mean Dice score."""
    model.eval()
    dices = torch.zeros(num_classes, device=device)
    counts = torch.zeros(num_classes, device=device)

    for imgs, masks in tqdm(loader, desc="Validation", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        preds = torch.argmax(logits, dim=1)

        for c in range(num_classes):
            p = (preds == c).float()
            t = (masks == c).float()
            inter = (p * t).sum()
            den = p.sum() + t.sum()
            if den > 0:
                dices[c] += (2 * inter / den)
                counts[c] += 1

    mean_dice = (dices[counts > 0] / counts[counts > 0]).mean().item() if (counts > 0).any() else 0.0
    return mean_dice

def fit(model, train_loader, val_loader, epochs=50, lr=1e-4, weight_decay=1e-5,
        num_classes=41, save_path="best_CADUNet3D.pt", accum_steps=1):
    """Train the model with OneCycle learning rate schedule."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = CombinedLossMultiClass(alpha=ALPHA_CE, beta=BETA_DICE)

    # OneCycle scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )

    best = -1
    history = {'train_loss': [], 'val_dice': [], 'lr': []}

    print("\n" + "="*60)
    print("Starting Training")
    print("="*60)

    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, accum_steps)
        val_dice = validate(model, val_loader, device, num_classes=num_classes)

        history['train_loss'].append(tr_loss)
        history['val_dice'].append(val_dice)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        print(f"Epoch {ep:03d}/{epochs} | train_loss={tr_loss:.4f} | val_dice={val_dice:.4f} | lr={optimizer.param_groups[0]['lr']:.6f}")

        if val_dice > best:
            best = val_dice
            torch.save({
                'epoch': ep,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': val_dice,
                'train_loss': tr_loss,
            }, save_path)
            print(f"  ✓ Saved best model (val_dice={best:.4f})")

        # Step scheduler after each batch (inside train_one_epoch is better, but this is simpler)
        for _ in range(len(train_loader)):
            scheduler.step()

    print("\n" + "="*60)
    print(f"Training Complete! Best Val Dice: {best:.4f}")
    print("="*60)

    return history

print("✓ Training functions defined")

✓ Training functions defined


In [ ]:
#@title Validation Function (CORRECTED — group-wise breakdown)
#
# Class groups for whole-brain CTA, derived from labelmap_topbrain_ct.txt:
#
#   MAJOR_ARTERIES (15 classes)
#     The Circle of Willis trunks. These are the clinically dominant vessels
#     and the ones most cerebrovascular papers report Dice on.
#       BA, R-/L-ICA, R-/L-M1/M2/M3, R-/L-A1A2/A3, R-/L-P1P2/P3P4
#     IDs: 1, 4-7, 11-14, 17-22
#
#   SMALL_ARTERIES (15 classes)
#     Communicating arteries, vertebrals, branch arteries, and anatomical
#     variants. Several are very thin (Acom, Pcom, OA) or only present in
#     a fraction of patients (3rd-A2, 3rd-A3 — azygos ACA variant).
#       Pcom, Acom, 3rd-A2/A3, VA, SCA, AICA, PICA, AChA, OA (L+R where applic.)
#     IDs: 2, 3, 8, 9, 10, 15, 16, 23-34
#
#   VEINS (6 classes)
#     Cerebral veins and dural venous sinuses. Different morphology from
#     arteries — typically larger but slower-flow, often less enhanced on
#     CTA, so segmentation behaviour differs systematically.
#       VoG, StS, ICVs, R-/L-BVR, SSS
#     IDs: 35-40
#
# Reporting these groups separately is far more informative than one macro
# average across all 40 — and far more defensible to a thesis examiner.

MAJOR_ARTERIES = [1, 4, 5, 6, 7, 11, 12, 13, 14, 17, 18, 19, 20, 21, 22]
SMALL_ARTERIES = [2, 3, 8, 9, 10, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
VEINS          = [35, 36, 37, 38, 39, 40]
ALL_FG         = MAJOR_ARTERIES + SMALL_ARTERIES + VEINS

# Anatomical variants that may be ABSENT in many volumes — log if missing
# but don't treat as a model failure (the new validation already excludes
# classes with zero GT voxels in the volume from per-class Dice).
ANATOMICAL_VARIANTS = [10, 15, 16]  # Acom, 3rd-A2, 3rd-A3

# For backwards compatibility with the rest of the notebook
MAJOR_VESSEL_CLASSES = MAJOR_ARTERIES

LAST_VAL_METRICS = {}


@torch.no_grad()
def validate_with_skeleton(model, loader, device, num_classes=41):
    """
    Returns (mean_macro_fg_dice, mean_macro_fg_skel) for the training loop's
    history dict — but populates LAST_VAL_METRICS with the full breakdown.

    Three group means are added on top of the per-class breakdown. All Dice
    scores are computed via global accumulation (intersections and cardinalities
    summed across the whole val set first, then Dice computed once per class).
    """
    global LAST_VAL_METRICS
    model.eval()

    # Per-class accumulators
    inter    = torch.zeros(num_classes, device=device, dtype=torch.float64)
    pred_sum = torch.zeros(num_classes, device=device, dtype=torch.float64)
    gt_sum   = torch.zeros(num_classes, device=device, dtype=torch.float64)
    skel_inter = torch.zeros(num_classes, device=device, dtype=torch.float64)
    skel_gt    = torch.zeros(num_classes, device=device, dtype=torch.float64)

    # Binary (vessel-vs-background) accumulators
    bin_i = bin_p = bin_g = 0.0

    for batch in tqdm(loader, desc="Validation", leave=False):
        if len(batch) == 3:
            imgs, masks, skels = batch
            skels = skels.to(device)
        else:
            imgs, masks = batch
        # No skeletons available — synthesize a dummy so skeleton-recall code
        # paths don't crash. Skeleton metrics will be 0 / meaningless for this run.
            skels = torch.zeros_like(masks)
    imgs  = imgs.to(device)
    masks = masks.to(device)
    if not torch.is_tensor(skels) or skels.device != device:
        skels = skels.to(device)

        logits = model(imgs)
        preds  = torch.argmax(logits, dim=1)
        probs  = F.softmax(logits, dim=1)

        for c in range(num_classes):
            p = (preds == c)
            t = (masks == c)
            inter[c]    += (p & t).sum().double()
            pred_sum[c] += p.sum().double()
            gt_sum[c]   += t.sum().double()

            ts = (skels == c).float()
            skel_inter[c] += (probs[:, c] * ts).sum().double()
            skel_gt[c]    += ts.sum().double()

        pv = (preds > 0)
        gv = (masks > 0)
        bin_i += (pv & gv).sum().item()
        bin_p += pv.sum().item()
        bin_g += gv.sum().item()

    eps = 1e-7

    per_class_dice = {}
    per_class_skel = {}
    per_class_voxels = {}
    for c in range(num_classes):
        per_class_voxels[c] = int(gt_sum[c].item())
        if gt_sum[c].item() > 0:
            per_class_dice[c] = (2 * inter[c] / (pred_sum[c] + gt_sum[c] + eps)).item()
            per_class_skel[c] = (skel_inter[c] / (skel_gt[c] + eps)).item()

    def _group_mean(class_ids, source):
        vals = [source[c] for c in class_ids if c in source]
        return float(np.mean(vals)) if vals else float('nan'), len(vals)

    major_dice, n_major = _group_mean(MAJOR_ARTERIES, per_class_dice)
    small_dice, n_small = _group_mean(SMALL_ARTERIES, per_class_dice)
    vein_dice,  n_vein  = _group_mean(VEINS,          per_class_dice)
    macro_fg_dice, _    = _group_mean(ALL_FG,         per_class_dice)

    major_skel, _ = _group_mean(MAJOR_ARTERIES, per_class_skel)
    small_skel, _ = _group_mean(SMALL_ARTERIES, per_class_skel)
    vein_skel,  _ = _group_mean(VEINS,          per_class_skel)
    macro_fg_skel, _ = _group_mean(ALL_FG,      per_class_skel)

    binary_dice = 2 * bin_i / (bin_p + bin_g + eps)

    LAST_VAL_METRICS = {
        'binary_dice':    binary_dice,
        'macro_fg_dice':  macro_fg_dice,
        'major_dice':     major_dice,    'major_present':  n_major,
        'small_dice':     small_dice,    'small_present':  n_small,
        'vein_dice':      vein_dice,     'vein_present':   n_vein,
        'macro_fg_skel':  macro_fg_skel,
        'major_skel':     major_skel,
        'small_skel':     small_skel,
        'vein_skel':      vein_skel,
        'per_class_dice': per_class_dice,
        'per_class_skel': per_class_skel,
        'per_class_voxels': per_class_voxels,
    }

    return macro_fg_dice, macro_fg_skel


print("✓ validate_with_skeleton (corrected, group-wise) defined")
print(f"  Major arteries : {len(MAJOR_ARTERIES)} classes  {MAJOR_ARTERIES}")
print(f"  Small arteries : {len(SMALL_ARTERIES)} classes  {SMALL_ARTERIES}")
print(f"  Veins/sinuses  : {len(VEINS)} classes  {VEINS}")

✓ validate_with_skeleton (corrected, group-wise) defined
  Major arteries : 15 classes  [1, 4, 5, 6, 7, 11, 12, 13, 14, 17, 18, 19, 20, 21, 22]
  Small arteries : 19 classes  [2, 3, 8, 9, 10, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
  Veins/sinuses  : 6 classes  [35, 36, 37, 38, 39, 40]


In [ ]:
#@title Inference Functions

@torch.no_grad()
def sliding_window_predict(volume_np, model, patch_size=(64, 128, 128),
                          overlap=(0.5, 0.5, 0.5), num_classes=41):
    """Predict on full volume using sliding window."""
    model.eval()
    D, H, W = volume_np.shape
    pd, ph, pw = patch_size
    sd = max(1, int(pd * (1 - overlap[0])))
    sh = max(1, int(ph * (1 - overlap[1])))
    sw = max(1, int(pw * (1 - overlap[2])))

    out_logits = np.zeros((num_classes, D, H, W), dtype=np.float32)
    out_counts = np.zeros((D, H, W), dtype=np.float32)

    total_patches = (math.ceil(D/sd) * math.ceil(H/sh) * math.ceil(W/sw))
    pbar = tqdm(total=total_patches, desc="Predicting")

    for z in range(0, max(1, D - pd + 1), sd):
        for y in range(0, max(1, H - ph + 1), sh):
            for x in range(0, max(1, W - pw + 1), sw):
                patch = volume_np[z:z+pd, y:y+ph, x:x+pw]

                if patch.shape != (pd, ph, pw):
                    pad_d = pd - patch.shape[0]
                    pad_h = ph - patch.shape[1]
                    pad_w = pw - patch.shape[2]
                    patch = np.pad(patch, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='edge')

                tens = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device)
                with autocast():
                    logits = model(tens)
                    logits = logits[0].cpu().float().numpy()

                dd, hh, ww = min(pd, D - z), min(ph, H - y), min(pw, W - x)
                out_logits[:, z:z+dd, y:y+hh, x:x+ww] += logits[:, :dd, :hh, :ww]
                out_counts[z:z+dd, y:y+hh, x:x+ww] += 1.0

                pbar.update(1)

    pbar.close()
    out_counts[out_counts == 0] = 1.0
    out_logits = out_logits / out_counts[None, ...]
    return out_logits

def save_nifti(mask_np, ref_img_path, out_path):
    """Save mask as NIfTI with reference header."""
    ref = nib.load(ref_img_path)
    nii = nib.Nifti1Image(mask_np.astype(np.int16), affine=ref.affine, header=ref.header)
    nib.save(nii, out_path)
    print(f"✓ Saved: {out_path}")

print("✓ Inference functions defined")

✓ Inference functions defined


In [ ]:
#@title Data Loading & Preparation

print("\n" + "="*60)
print("Loading Dataset")
print("="*60)

# Get file lists
img_pattern = os.path.join(DATA_DIR, IMG_GLOB)
msk_pattern = os.path.join(DATA_DIR, MSK_GLOB)

img_files = sorted(glob.glob(img_pattern))
msk_files = sorted(glob.glob(msk_pattern))

print(f"Found {len(img_files)} images and {len(msk_files)} masks")

if len(img_files) == 0:
    print("⚠ WARNING: No image files found! Check DATA_DIR and IMG_GLOB paths.")
    print(f"  Looking for: {img_pattern}")
else:
    print(f"✓ Sample image: {os.path.basename(img_files[0])}")

if len(msk_files) == 0:
    print("⚠ WARNING: No mask files found! Check DATA_DIR and MSK_GLOB paths.")
    print(f"  Looking for: {msk_pattern}")
else:
    print(f"✓ Sample mask: {os.path.basename(msk_files[0])}")

assert len(img_files) == len(msk_files), "Mismatch between images and masks count!"
assert len(img_files) > 0, "No data files found!"

# Train/val split
n_total = len(img_files)
n_train = int(n_total * TRAIN_VAL_SPLIT)

indices = list(range(n_total))
random.shuffle(indices)

train_indices = indices[:n_train]
val_indices = indices[n_train:]

train_imgs = [img_files[i] for i in train_indices]
train_msks = [msk_files[i] for i in train_indices]
val_imgs = [img_files[i] for i in val_indices]
val_msks = [msk_files[i] for i in val_indices]

print(f"\nSplit: {len(train_imgs)} training, {len(val_imgs)} validation")

# Create datasets
print("\nCreating datasets...")
train_ds = TopBrainCT3D(
    train_imgs, train_msks,
    patch_size=PATCH_SIZE,
    augment=True,
    foreground_sampling=True,
    num_classes=NUM_CLASSES,
    cache_volumes=True,
    patches_per_volume=10
)

val_ds = TopBrainCT3D(
    val_imgs, val_msks,
    patch_size=PATCH_SIZE,
    augment=False,
    foreground_sampling=False,
    num_classes=NUM_CLASSES,
    cache_volumes=True,
    patches_per_volume=5
)

print(f"✓ Training dataset: {len(train_ds)} patches")
print(f"✓ Validation dataset: {len(val_ds)} patches")

# Create dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✓ Training batches per epoch: {len(train_loader)}")
print(f"✓ Validation batches: {len(val_loader)}")


Loading Dataset
Found 40 images and 40 masks
✓ Sample image: topcow_ct_001_0000.nii.gz
✓ Sample mask: topcow_ct_001.nii.gz

Split: 32 training, 8 validation

Creating datasets...
✓ Training dataset: 320 patches
✓ Validation dataset: 40 patches
✓ Training batches per epoch: 160
✓ Validation batches: 20


In [ ]:
#@title Initialize Model

print("\n" + "="*60)
print("Initializing Model")
print("="*60)

model = CADUNet3D(in_ch=1, base=BASE_CH, num_classes=NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model: CAD U-Net 3D")
print(f"  Base channels: {BASE_CH}")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device: {device}")


Initializing Model
✓ Model: CAD U-Net 3D
  Base channels: 32
  Number of classes: 41
  Total parameters: 23,677,091
  Trainable parameters: 23,677,091
  Device: cuda


In [ ]:
#@title Train Model

print("\n" + "="*60)
print("Configuration Summary")
print("="*60)
print(f"Patch size: {PATCH_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation steps: {ACCUM_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate (max): {LR_MAX}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Loss weights: CE={ALPHA_CE}, Dice={BETA_DICE}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print("="*60)

history = fit(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR_MAX,
    weight_decay=WEIGHT_DECAY,
    num_classes=NUM_CLASSES,
    save_path=CHECKPOINT_PATH,
    accum_steps=ACCUM_STEPS
)

print("\n⚠ Training is commented out by default.")
print("Uncomment the 'history = fit(...)' block above to start training.")

In [ ]:
#@title Plot Training History (Run after training)

def plot_training_history(history, save_path=None):
    """Plot training loss and validation dice over epochs."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

    epochs = range(1, len(history['train_loss']) + 1)

    # Training loss
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True, alpha=0.3)

    # Validation Dice
    ax2.plot(epochs, history['val_dice'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Dice')
    ax2.set_title('Validation Dice Score')
    ax2.grid(True, alpha=0.3)

    # Learning rate
    ax3.plot(epochs, history['lr'], 'r-', linewidth=2)
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Learning Rate')
    ax3.set_title('Learning Rate (OneCycle)')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved plot to {save_path}")

    plt.show()

# Uncomment after training:
# plot_training_history(history, save_path=os.path.join(RESULTS_DIR, "training_history.png"))

In [ ]:
#@title Inference on New Volume

def predict_volume(img_path, model_path, output_path,
                   patch_size=PATCH_SIZE, overlap=(0.5, 0.5, 0.5)):
    """Complete inference pipeline for a single volume."""
    print("\n" + "="*60)
    print("Running Inference")
    print("="*60)
    print(f"Input: {img_path}")
    print(f"Model: {model_path}")
    print(f"Output: {output_path}")

    # Load volume
    print("\n1. Loading volume...")
    vol = nib.load(img_path).get_fdata().astype(np.float32)
    print(f"   Volume shape: {vol.shape}")

    # Normalize
    print("2. Normalizing...")
    vol = (vol - vol.mean()) / (vol.std() + 1e-6)

    # Load model
    print("3. Loading model...")
    model = CADUNet3D(in_ch=1, base=BASE_CH, num_classes=NUM_CLASSES).to(device)
    checkpoint = torch.load(model_path, map_location=device)

    # Fix for DataParallel checkpoints
    state_dict = checkpoint['model_state_dict']
    if any(k.startswith('module.') for k in state_dict.keys()):
        print("   Detected 'module.' prefix in checkpoint → cleaning keys...")
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)

    model.eval()
    print(f"   Loaded checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")
    print(f"   Validation Dice: {checkpoint.get('val_dice', 'unknown')}")

    # Predict
    print("4. Running sliding window prediction...")
    logits_vol = sliding_window_predict(
        vol, model,
        patch_size=patch_size,
        overlap=overlap,
        num_classes=NUM_CLASSES
    )

    # Get prediction
    print("5. Computing final segmentation...")
    pred_mask = np.argmax(logits_vol, axis=0).astype(np.int16)

    # Count labels
    unique, counts = np.unique(pred_mask, return_counts=True)
    print(f"   Predicted labels: {len(unique)}")
    for label, count in zip(unique, counts):
        pct = 100 * count / pred_mask.size
        if pct > 0.1:  # Only show labels with >0.1% presence
            print(f"     Label {label}: {count} voxels ({pct:.2f}%)")

    # Save
    print("6. Saving result...")
    save_nifti(pred_mask, img_path, output_path)

    print("\n" + "="*60)
    print("✓ Inference Complete!")
    print("="*60)

    return pred_mask

# Example usage (uncomment and modify paths after training):
test_img = "/content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation/imagesTr_topbrain_ct/topcow_ct_001_0000.nii.gz"
output_pred = os.path.join(RESULTS_DIR, "prediction.nii.gz")
pred = predict_volume(test_img, CHECKPOINT_PATH, output_pred)


Running Inference
Input: /content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation/imagesTr_topbrain_ct/topcow_ct_001_0000.nii.gz
Model: /content/drive/MyDrive/3D_CAD_Unet/Model/best_CADUNet3D.pt
Output: /content/drive/MyDrive/3D_CAD_Unet/Results/prediction.nii.gz

1. Loading volume...
   Volume shape: (284, 327, 243)
2. Normalizing...
3. Loading model...
   Detected 'module.' prefix in checkpoint → cleaning keys...
   Loaded checkpoint from epoch 70
   Validation Dice: 0.09403210133314133
4. Running sliding window prediction...


Predicting:   0%|          | 0/216 [00:00<?, ?it/s]/tmp/ipykernel_3404/2363441414.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Predicting:  26%|██▌       | 56/216 [00:28<01:21,  1.96it/s]


5. Computing final segmentation...
   Predicted labels: 8
     Label 0: 22547867 voxels (99.92%)
6. Saving result...
✓ Saved: /content/drive/MyDrive/3D_CAD_Unet/Results/prediction.nii.gz

✓ Inference Complete!


In [ ]:
#@title Visualize Predictions

def visualize_prediction(img_path, mask_path, pred_path=None, slice_idx=None, save_path=None):
    """Visualize image, ground truth mask, and prediction."""
    img = nib.load(img_path).get_fdata()
    mask = nib.load(mask_path).get_fdata()

    if slice_idx is None:
        slice_idx = img.shape[0] // 2  # Middle slice

    fig, axes = plt.subplots(1, 3 if pred_path else 2, figsize=(15, 5))

    # Image
    axes[0].imshow(img[slice_idx], cmap='gray')
    axes[0].set_title(f'Image (slice {slice_idx})')
    axes[0].axis('off')

    # Ground truth
    axes[1].imshow(mask[slice_idx], cmap='nipy_spectral', vmin=0, vmax=NUM_CLASSES-1)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')

    # Prediction
    if pred_path:
        pred = nib.load(pred_path).get_fdata()
        axes[2].imshow(pred[slice_idx], cmap='nipy_spectral', vmin=0, vmax=NUM_CLASSES-1)
        axes[2].set_title('Prediction')
        axes[2].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved visualization to {save_path}")

    plt.show()

# Example usage:
# visualize_prediction(
#     img_path=test_img,
#     mask_path=test_mask,
#     pred_path=output_pred,
#     slice_idx=32,
#     save_path=os.path.join(RESULTS_DIR, "visualization.png")
# )

In [ ]:
#@title Compute Metrics

@torch.no_grad()
def compute_detailed_metrics(pred_path, gt_path, num_classes=NUM_CLASSES):
    """Compute detailed per-class metrics."""
    pred = nib.load(pred_path).get_fdata().astype(np.int64)
    gt = nib.load(gt_path).get_fdata().astype(np.int64)

    print("\n" + "="*60)
    print("Per-Class Metrics")
    print("="*60)
    print(f"{'Class':<6} {'Dice':<8} {'IoU':<8} {'Precision':<10} {'Recall':<8}")
    print("-" * 60)

    dices = []
    ious = []

    for c in range(num_classes):
        pred_c = (pred == c)
        gt_c = (gt == c)

        tp = np.sum(pred_c & gt_c)
        fp = np.sum(pred_c & ~gt_c)
        fn = np.sum(~pred_c & gt_c)

        if tp + fp + fn == 0:
            continue

        dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        dices.append(dice)
        ious.append(iou)

        print(f"{c:<6} {dice:<8.4f} {iou:<8.4f} {precision:<10.4f} {recall:<8.4f}")

    print("-" * 60)
    print(f"{'Mean':<6} {np.mean(dices):<8.4f} {np.mean(ious):<8.4f}")
    print("=" * 60)

    return {
        'mean_dice': np.mean(dices),
        'mean_iou': np.mean(ious),
        'per_class_dice': dices,
        'per_class_iou': ious
    }

# Example usage:
# metrics = compute_detailed_metrics(output_pred, test_mask)

In [ ]:
#@title Batch Inference

def batch_inference(img_dir, output_dir, model_path, img_pattern="*.nii.gz"):
    """Run inference on all images in a directory."""
    os.makedirs(output_dir, exist_ok=True)

    img_files = sorted(glob.glob(os.path.join(img_dir, img_pattern)))
    print(f"\nFound {len(img_files)} images for inference")

    for img_path in img_files:
        basename = os.path.basename(img_path)
        output_path = os.path.join(output_dir, basename.replace('.nii.gz', '_pred.nii.gz'))

        try:
            predict_volume(img_path, model_path, output_path)
        except Exception as e:
            print(f"✗ Error processing {basename}: {e}")
            continue

    print(f"\n✓ Batch inference complete. Results saved to {output_dir}")

# Example usage:
# batch_inference(
#     img_dir="/content/drive/MyDrive/test_images/",
#     output_dir=os.path.join(RESULTS_DIR, "batch_predictions"),
#     model_path=CHECKPOINT_PATH
# )

In [ ]:
#@title Save Configuration

def save_config(save_path=None):
    """Save training configuration to file."""
    if save_path is None:
        save_path = os.path.join(RESULTS_DIR, "config.txt")

    config = f"""CAD U-Net 3D Training Configuration
{'='*60}
Data:
  DATA_DIR: {DATA_DIR}
  IMG_GLOB: {IMG_GLOB}
  MSK_GLOB: {MSK_GLOB}
  LABELMAP_TXT: {LABELMAP_TXT}
  NUM_CLASSES: {NUM_CLASSES}

Model:
  Architecture: CAD U-Net 3D
  BASE_CH: {BASE_CH}
  PATCH_SIZE: {PATCH_SIZE}

Training:
  BATCH_SIZE: {BATCH_SIZE}
  ACCUM_STEPS: {ACCUM_STEPS}
  Effective batch size: {BATCH_SIZE * ACCUM_STEPS}
  EPOCHS: {EPOCHS}
  LR_MAX: {LR_MAX}
  WEIGHT_DECAY: {WEIGHT_DECAY}

Loss:
  ALPHA_CE: {ALPHA_CE}
  BETA_DICE: {BETA_DICE}

Other:
  FOREGROUND_SAMPLING_PROB: {FOREGROUND_SAMPLING_PROB}
  TRAIN_VAL_SPLIT: {TRAIN_VAL_SPLIT}
  RANDOM_SEED: {RANDOM_SEED}
  RESULTS_DIR: {RESULTS_DIR}
  CHECKPOINT_PATH: {CHECKPOINT_PATH}

Device: {device}
{'='*60}
"""

    with open(save_path, 'w') as f:
        f.write(config)

    print(f"✓ Configuration saved to {save_path}")
    return config

# Save configuration
config_text = save_config()
print("\n" + config_text)

# ============================================================================
print("\n" + "="*60)
print("✓ All components loaded successfully!")
print("="*60)
print("\nNext steps:")
print("1. Verify your data paths are correct")
print("2. Uncomment the training code in the 'Train Model' section")
print("3. Run training: history = fit(...)")
print("4. Monitor training progress")
print("5. Use inference functions to predict on new volumes")
print("\nFor questions or issues, check the configuration summary above.")
print("="*60)

In [ ]:
#@title Show all class names (run this once to find your major-vessel IDs)
# Look for "ICA", "MCA", "ACA", "PCA", "BA", "VA", "AcomA", "PcomA" in the output,
# then put their idx numbers into MAJOR_VESSEL_CLASSES at the top of cell 13.

if 'LABEL_INFO' not in globals() or len(LABEL_INFO) == 0:
    print("LABEL_INFO is empty — your labelmap file did not load. Check LABELMAP_TXT path.")
else:
    print(f"{'idx':>3}  {'name':<40}  rgb")
    print("-" * 70)
    for idx in sorted(LABEL_INFO):
        entry = LABEL_INFO[idx]
        nm  = entry.get('name') or '(unnamed)'
        rgb = entry.get('rgb')  or ''
        print(f"{idx:>3}  {nm:<40}  {rgb}")
    print()
    print("→ Update MAJOR_VESSEL_CLASSES in cell 13 with the IDs of your major arteries,")
    print("  then re-run cell 13 followed by the next cell to get correct numbers.")

idx  name                                      rgb
----------------------------------------------------------------------
  0  (unnamed)                                 (0, 0, 0)
  1  (unnamed)                                 (182, 1, 1)
  2  (unnamed)                                 (255, 1, 1)
  3  (unnamed)                                 (66, 1, 1)
  4  (unnamed)                                 (190, 1, 1)
  5  (unnamed)                                 (193, 1, 1)
  6  (unnamed)                                 (152, 1, 1)
  7  (unnamed)                                 (253, 1, 1)
  8  (unnamed)                                 (113, 1, 1)
  9  (unnamed)                                 (92, 1, 1)
 10  (unnamed)                                 (66, 1, 1)
 11  (unnamed)                                 (255, 1, 1)
 12  (unnamed)                                 (2, 1, 1)
 13  (unnamed)                                 (0, 1, 1)
 14  (unnamed)                                 (0, 1, 1)
 15 

In [ ]:
#@title Re-evaluate best checkpoint with corrected metrics (group-wise)
import os, torch, numpy as np, json

assert os.path.exists(CHECKPOINT_PATH), \
    f"No checkpoint found at {CHECKPOINT_PATH}. Train the model first."

# ---- Load + remap checkpoint keys ----------------------------------------
# Handles three things that may differ between the saved checkpoint and the
# current CADUNet3D class definition:
#   1. DataParallel-wrapped checkpoints prefix every key with "module."
#   2. Older versions used `head.*` instead of `head_mc.*`
#   3. Older versions used `uN.ag.Wg.*` / `uN.ag.Wx.*` instead of
#      `uN.attn.W_g.*` / `uN.attn.W_x.*`

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
state_dict = ckpt['model_state_dict']

def remap_key(k):
    # Strip DataParallel "module." prefix
    if k.startswith('module.'):
        k = k[len('module.'):]
    # head.* → head_mc.*
    if k.startswith('head.'):
        return 'head_mc.' + k[len('head.'):]
    # uN.ag.Wg.* / Wx.* / psi.* → uN.attn.W_g.* / W_x.* / psi.*
    for n in ('u1', 'u2', 'u3', 'u4'):
        prefix = f'{n}.ag.'
        if k.startswith(prefix):
            rest = k[len(prefix):]
            rest = rest.replace('Wg.', 'W_g.', 1)
            rest = rest.replace('Wx.', 'W_x.', 1)
            return f'{n}.attn.{rest}'
    return k

remapped = {remap_key(k): v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(remapped, strict=False)
if missing:
    print(f"  WARNING: {len(missing)} missing keys (showing first 5):")
    for k in missing[:5]: print(f"    {k}")
if unexpected:
    print(f"  WARNING: {len(unexpected)} unexpected keys (showing first 5):")
    for k in unexpected[:5]: print(f"    {k}")
if not missing and not unexpected:
    print("  ✓ all weights matched cleanly")

model.eval()
print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")
print(f"  saved val_dice (old metric) : {ckpt.get('val_dice', float('nan')):.4f}")
print(f"  saved val_skel (old metric) : {ckpt.get('val_skel', float('nan')):.4f}\n")

# ---- Run corrected validation --------------------------------------------
_ = validate_with_skeleton(model, val_loader, device, num_classes=NUM_CLASSES)
m = LAST_VAL_METRICS

print("=" * 72)
print("  CORRECTED VALIDATION METRICS (group-wise)")
print("=" * 72)
print(f"  Binary Dice (vessel vs background)   : {m['binary_dice']:.4f}   ← thesis headline")
print(f"  Macro foreground Dice (40 fg cls)    : {m['macro_fg_dice']:.4f}")
print()
print(f"  {'Group':<22} {'#cls':>6} {'Dice':>8} {'SkelRecall':>12}")
print("  " + "-" * 54)
print(f"  {'Major arteries':<22} {m['major_present']:>6} {m['major_dice']:>8.4f} {m['major_skel']:>12.4f}")
print(f"  {'Small arteries':<22} {m['small_present']:>6} {m['small_dice']:>8.4f} {m['small_skel']:>12.4f}")
print(f"  {'Veins/sinuses':<22} {m['vein_present']:>6} {m['vein_dice']:>8.4f} {m['vein_skel']:>12.4f}")
print("=" * 72)

# Per-class breakdown — sorted ascending so missing classes are at the top
print("\nPer-class Dice (foreground only, ascending):")
print(f"  {'idx':>3}  {'name':<20} {'GT_vox':>10}  {'Dice':>7}  {'SkelRec':>8}  group")
print("  " + "-" * 70)

GROUP_TAG = {}
for c in MAJOR_ARTERIES: GROUP_TAG[c] = 'major'
for c in SMALL_ARTERIES: GROUP_TAG[c] = 'small'
for c in VEINS:          GROUP_TAG[c] = 'vein'

items = [(c, d) for c, d in m['per_class_dice'].items() if c != 0]
items.sort(key=lambda x: x[1])

for c, dice in items:
    nm = LABEL_INFO.get(c, {}).get('name') or '?'
    if nm == '?' or nm is None:
        nm = f'(idx {c})'
    sk = m['per_class_skel'].get(c, float('nan'))
    vox = m['per_class_voxels'].get(c, 0)
    grp = GROUP_TAG.get(c, '?')
    print(f"  {c:>3}  {nm[:20]:<20} {vox:>10,}  {dice:>7.4f}  {sk:>8.4f}  {grp}")

# Note variants that were absent in the validation set
absent_variants = [c for c in ANATOMICAL_VARIANTS
                   if c not in m['per_class_dice']]
if absent_variants:
    print(f"\n  Note: anatomical variants absent in val set "
          f"(no GT voxels): {absent_variants}")
    print("        ↑ these are normal anatomical variations missing in some patients,")
    print("          not model failures.")

# Save
out_path = os.path.join(RESULTS_DIR, 'corrected_metrics.json')
os.makedirs(RESULTS_DIR, exist_ok=True)
to_save = {}
for k, v in m.items():
    if isinstance(v, dict):
        to_save[k] = {str(kk): vv for kk, vv in v.items()}
    else:
        to_save[k] = v
to_save['groups'] = {
    'major_arteries': MAJOR_ARTERIES,
    'small_arteries': SMALL_ARTERIES,
    'veins':          VEINS,
}
to_save['checkpoint_epoch'] = ckpt.get('epoch')
with open(out_path, 'w') as f:
    json.dump(to_save, f, indent=2)
print(f"\nFull breakdown saved to: {out_path}")

  ✓ all weights matched cleanly
Loaded checkpoint from epoch 70
  saved val_dice (old metric) : 0.0940
  saved val_skel (old metric) : nan



  CORRECTED VALIDATION METRICS (group-wise)
  Binary Dice (vessel vs background)   : 0.4164   ← thesis headline
  Macro foreground Dice (40 fg cls)    : 0.0789

  Group                    #cls     Dice   SkelRecall
  ------------------------------------------------------
  Major arteries              3   0.0478       0.0000
  Small arteries              6   0.0000       0.0000
  Veins/sinuses               3   0.2676       0.0000

Per-class Dice (foreground only, ascending):
  idx  name                     GT_vox     Dice   SkelRec  group
  ----------------------------------------------------------------------
    2  (idx 2)                       3   0.0000    0.0000  small
    3  (idx 3)                     945   0.0000    0.0000  small
   22  (idx 22)                    391   0.0000    0.0000  major
   23  (idx 23)                    442   0.0000    0.0000  small
   24  (idx 24)                  2,878   0.0000    0.0000  small
   25  (idx 25)                      4   0.0000    0.0000

In [ ]:
#@title Generate Table 3 row for the thesis (group-wise, LaTeX-ready)
m = LAST_VAL_METRICS

print("Spreadsheet-paste version (tab-separated):")
print(f"Method\tBinary\tMajor\tSmall\tVein\tMacro-fg\tSkelRecall")
print(
    f"Adaptive Skeleton-Aware (proposed)"
    f"\t{m['binary_dice']:.3f}"
    f"\t{m['major_dice']:.3f}"
    f"\t{m['small_dice']:.3f}"
    f"\t{m['vein_dice']:.3f}"
    f"\t{m['macro_fg_dice']:.3f}"
    f"\t{m['macro_fg_skel']:.3f}"
)
print()

print("LaTeX line:")
print(
    "Adaptive Skeleton-Aware (proposed) & "
    f"{m['binary_dice']:.3f} & "
    f"{m['major_dice']:.3f} & "
    f"{m['small_dice']:.3f} & "
    f"{m['vein_dice']:.3f} & "
    f"{m['macro_fg_dice']:.3f} & "
    f"{m['macro_fg_skel']:.3f} \\\\"
)
print()
print("Suggested LaTeX header:")
print(
    "Method & Binary & Major arteries & Small arteries & Veins/sinuses & "
    "Macro-fg & Skel.\\,Recall \\\\"
)


In [ ]:
# Diagnostic: what's actually in the val_loader right now?
import numpy as np
print(f"len(val_imgs): {len(val_imgs)}")
print(f"len(val_loader): {len(val_loader)} batches")
print(f"val_ds.patches_per_volume: {val_ds.patches_per_volume}")
print(f"val_ds.patch_size: {val_ds.patch_size}")
print(f"val_ds.fg_sampling: {val_ds.fg_sampling}")
print()

# Which volume files are in val?
print("Val volumes:")
for f in val_imgs:
    print(f"  {os.path.basename(f)}")

len(val_imgs): 8
len(val_loader): 20 batches
val_ds.patches_per_volume: 5
val_ds.patch_size: (64, 128, 128)
val_ds.fg_sampling: False

Val volumes:
  topcow_ct_026_0000.nii.gz
  topcow_ct_005_0000.nii.gz
  topcow_ct_029_0000.nii.gz
  topcow_ct_012_0000.nii.gz
  topcow_ct_011_0000.nii.gz
  topcow_ct_008_0000.nii.gz
  topcow_ct_004_0000.nii.gz
  topcow_ct_006_0000.nii.gz
